In [1]:
import numpy as np
import pandas as pd
from types import resolve_bases
import pickle
import plotly.express as px
from SamplingMethods import Sampler_class
from ax.api.client import Client
from ax.api.configs import RangeParameterConfig
from ax.generation_strategy.center_generation_node import CenterGenerationNode
from ax.generation_strategy.transition_criterion import MinTrials
from ax.generation_strategy.generation_strategy import GenerationStrategy
from ax.generation_strategy.generation_node import GenerationNode
from ax.generation_strategy.model_spec import GeneratorSpec
from ax.modelbridge.registry import Generators
from gpytorch.kernels import MaternKernel
from botorch.models import SingleTaskGP
from botorch.models.transforms.input import Warp
from botorch.models.map_saas import AdditiveMapSaasSingleTaskGP
from ax.utils.stats.model_fit_stats import MSE
from ax.models.torch.botorch_modular.surrogate import SurrogateSpec, ModelConfig
from botorch.acquisition.logei import qLogNoisyExpectedImprovement

In [2]:
client = Client()
gp_model = client.load_from_json_file("/Users/thomasdodd/Library/CloudStorage/OneDrive-MillfieldEnterprisesLimited/Cambridge/PhD/writing/papers/UoC_Paper1/Sandbox/Modelling/ModelMk16.json")
gp_model.get_next_trials(max_trials=1)
def SurrogateModelOfReality(s1, s2, b1):
    y_pred = gp_model.predict([{"s1":s1,"s2":s2,"b1":b1}])[0]["t1"][0]
    return np.float64(y_pred)

/Users/thomasdodd/miniconda3/envs/ax1_env/lib/python3.12/site-packages/botorch/optim/optimize.py:677: RuntimeWarning: Optimization failed in `gen_candidates_scipy` with the following warning(s):
[OptimizationWarning('Optimization failed within `scipy.optimize.minimize` with status 2 and message ABNORMAL: .')]
Trying again with a new set of initial conditions.
  return _optimize_acqf_batch(opt_inputs=opt_inputs)


In [3]:
class OptimisationSetup_class(object):
    def __init__(self):
        self.Parameters_lis = [
            RangeParameterConfig(name="s1", parameter_type="float", bounds=(0, 1)),
            RangeParameterConfig(name="s2", parameter_type="float", bounds=(0, 1)),
            RangeParameterConfig(name="b1", parameter_type="float", bounds=(0, 1)),
        ]
OptimisationSetup_obj = OptimisationSetup_class()

In [4]:
y_max_lis = []

for i in range(100):
    client = Client()
    parameters = [
        RangeParameterConfig(
            name="s1", parameter_type="float", bounds=(0, 1)
        ),
        RangeParameterConfig(
            name="s2", parameter_type="float", bounds=(0, 1)
        ),
        RangeParameterConfig(
            name="b1", parameter_type="float", bounds=(0, 1)
        ),
    ]
    client.configure_experiment(parameters=parameters)
    def construct_generation_strategy(
        generator_spec: GeneratorSpec, node_name: str,
    ) -> GenerationStrategy:
        """Constructs a Center + Sobol + Modular BoTorch `GenerationStrategy`
        using the provided `generator_spec` for the Modular BoTorch node.
        """
        botorch_node = GenerationNode(
            node_name=node_name,
            model_specs=[generator_spec],
        )
        return GenerationStrategy(
            name=f"{node_name}",
            nodes=[botorch_node]
        )

    # Let's construct the simplest version with all defaults.
    construct_generation_strategy(
        generator_spec=GeneratorSpec(model_enum=Generators.BOTORCH_MODULAR),
        node_name="Modular BoTorch",
    )

    surrogate_spec = SurrogateSpec(
        model_configs=[
            # Select between two models:
            # An additive mixture of relatively strong SAAS priors with input Warping.
            # A relatively vanilla GP with a Matern kernel.
            ModelConfig(
                botorch_model_class=SingleTaskGP,
                covar_module_class=MaternKernel,
                covar_module_options={"nu": 2.5},
            ),
        ],
        eval_criterion=MSE,  # Select the model to use as the one that minimizes mean squared error.
        allow_batched_models=False,  # Forces each metric to be modeled with an independent BoTorch model.
        # If we wanted to specify different options for different metrics.
        # metric_to_model_configs: dict[str, list[ModelConfig]]
    )

    generator_spec = GeneratorSpec(
        model_enum=Generators.BOTORCH_MODULAR,
        model_kwargs={
            "surrogate_spec": surrogate_spec,
            "botorch_acqf_class": qLogNoisyExpectedImprovement,
            # Can be used for additional inputs that are not constructed
            # by default in Ax. We will demonstrate below.
            "acquisition_options": {},
        },
        # We can specify various options for the optimizer here.
        model_gen_kwargs = {
            "model_gen_options": {
                "optimizer_kwargs": {
                    "num_restarts": 20,
                    "sequential": False,
                    "options": {
                        "batch_limit": 5,
                        "maxiter": 200,
                    },
                },
            },
        }
    )

    generation_strategy = construct_generation_strategy(
        generator_spec=generator_spec,
        node_name="BoTorch w/ Model Selection",
    )
    generation_strategy

    client.set_generation_strategy(
        generation_strategy=generation_strategy,
    )

    metric_name = "t1" # this name is used during the optimization loop in Step 5
    objective = f"{metric_name}" # minimization is specified by the negative sign

    client.configure_optimization(objective=objective)

    # Quasirandom Sampling Exercise
    sampler_obj = Sampler_class()
    Parameters_lis = [
        {"name":"s1", "type":"range","bounds":[0,1],"value_type":"float"},
        {"name":"s2", "type":"range","bounds":[0,1],"value_type":"float"},
        {"name":"b1", "type":"range","bounds":[0,1],"value_type":"float"}
    ]
    X = sampler_obj.three.QuasirandomSampler3D_func(8,Parameters_lis).T

    for array in X:
        my_parameters = {"s1": array[0], "s2": array[1], "b1": array[2]}
        trial_index = client.attach_trial(parameters=my_parameters)
        client.complete_trial(trial_index=trial_index,raw_data={"t1": SurrogateModelOfReality(**my_parameters)})

    for _ in range(21): # Run 21 rounds of trials
        trial = sampler_obj.McIntersiteProjTh.McIntersiteProjTh_func(OptimisationSetup_obj,client)
        s1 = trial[0][0]
        s2 = trial[0][1]
        b1 = trial[0][2]
        parameters = {"s1":s1,"s2":s2,"b1":b1}
        trial_index = client.attach_trial(parameters=parameters)
        result = SurrogateModelOfReality(s1,s2,b1)
        raw_data = {metric_name: result}
        client.complete_trial(trial_index=trial_index, raw_data=raw_data)
    client._experiment.trials.pop(28)
    client._experiment.trials.pop(27)
    print(f"Trial {i} =========================================")
    y_max = np.max(np.array(client.summarize().t1))
    print(y_max)
    y_max_lis.append(y_max)
    print()

y_max_arr = np.array(y_max_lis)
print(y_max_arr)

Trial 0 =========================================
14.835105613371711

Trial 1 =========================================
16.52169983930649

Trial 2 =========================================
13.69258130852648

Trial 3 =========================================
14.000993466061566

Trial 4 =========================================
15.070785748817595

Trial 5 =========================================
14.492626084890713

Trial 6 =========================================
15.66408252669877

Trial 7 =========================================
13.850087009794894

Trial 8 =========================================
13.889515806065862

Trial 9 =========================================
13.941315352520528

Trial 10 =========================================
14.366609991658366

Trial 11 =========================================
14.491104243557633

Trial 12 =========================================
14.39749845226483

Trial 13 =========================================
14.748902648775616

Trial 14 ===========

In [5]:
print(f"Max = {np.max(y_max_arr)}")
print(f"Avg = {np.average(y_max_arr)}")
print(f"Std = {np.std(y_max_arr)}")

Max = 17.151411032526514
Avg = 14.540814682786399
Std = 0.6714849826782195


In [6]:
print(y_max_arr.tolist())

[14.835105613371711, 16.52169983930649, 13.69258130852648, 14.000993466061566, 15.070785748817595, 14.492626084890713, 15.66408252669877, 13.850087009794894, 13.889515806065862, 13.941315352520528, 14.366609991658366, 14.491104243557633, 14.39749845226483, 14.748902648775616, 13.55904689429837, 14.369355206820682, 15.964448024319147, 13.731006124814753, 14.117903565246998, 16.014763437827895, 14.002358793815329, 15.846895201533354, 14.05210229572971, 14.292227153932672, 14.219632840158106, 14.093300224204967, 13.639152508810946, 15.866633308434633, 15.188676443658329, 17.151411032526514, 15.794246451756093, 13.912874699036301, 13.994932560945399, 13.990927773581824, 15.193989674261397, 14.942222425374428, 15.476427107524628, 14.625426767398233, 14.346530574749998, 14.285382328088698, 14.2594239298258, 14.52235369533637, 14.727644428469869, 14.755331223891364, 13.799469215138306, 14.764524902696024, 14.020200999118597, 14.133941578542258, 14.315321792365676, 15.498117719624544, 13.62867

In [7]:
filepath = "/Users/thomasdodd/Library/CloudStorage/OneDrive-MillfieldEnterprisesLimited/Cambridge/PhD/writing/papers/UoC_Paper1/Sandbox/SequentialTestswGPModel/DataGenerated/normal_MIPT_9_27_3.pkl"
loadeddf = pd.read_pickle(filepath_or_buffer=filepath)
latestdf = pd.DataFrame(y_max_arr)
newdf = pd.concat(objs=[loadeddf,latestdf],axis=0)
newdf = newdf.reset_index(drop=True)
pd.to_pickle(obj=newdf,filepath_or_buffer=filepath)

In [8]:
filepath = "/Users/thomasdodd/Library/CloudStorage/OneDrive-MillfieldEnterprisesLimited/Cambridge/PhD/writing/papers/UoC_Paper1/sandbox/SequentialTestswGPModel/DataGenerated/normal_MIPT_9_27_3.pkl"
loadeddf = pd.read_pickle(filepath_or_buffer=filepath)
print(loadeddf)
# newdf = loadeddf.drop(loadeddf.index, inplace=True)
# pd.to_pickle(obj=newdf,filepath_or_buffer=filepath)
# print(newdf)

             0
0    15.191125
1    15.592263
2    14.725177
3    15.554784
4    15.190057
..         ...
595  14.675504
596  14.055032
597  14.445411
598  13.982052
599  14.884276

[600 rows x 1 columns]


In [9]:
# # Sanity check to make sure the MIPT is running correctly.
# df = client.summarize()
# types_lis = []
# for i in range(len(df)):
#     if i < 8:
#         types_lis.append("one-shot")
#     else:
#         types_lis.append("sequential")
# df["type"] = types_lis
# fig = px.scatter_3d(df, x='s1', y='s2', z='b1', color='type',width=1300, height=600)
# fig.show()